# Fundamentals of Machine Learning - Exercise 9
Goal of this excercise is to complete the hands-on experience of the classification task.

## Household Prices Dataset
https://www.kaggle.com/c/house-prices-advanced-regression-techniques/data

* ... I bet that you already know the data pretty well 😅

![meme03](https://github.com/rasvob/VSB-FEI-Fundamentals-of-Machine-Learning-Exercises/blob/master/images/fml_09_meme_03.jpg?raw=true)

**Important attributes description:**
* SalePrice: The property's sale price in dollars. This is the target variable that you're trying to predict.
* MSSubClass: The building class
* BldgType: Type of dwelling
* HouseStyle: Style of dwelling
* OverallQual: Overall material and finish quality
* OverallCond: Overall condition rating
* YearBuilt: Original construction date
* Heating: Type of heating
* CentralAir: Central air conditioning
* GrLivArea: Above grade (ground) living area square feet
* BedroomAbvGr: Number of bedrooms above basement level)



In [1]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import math

from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split, StratifiedKFold, KFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, recall_score, precision_score, confusion_matrix, auc
from sklearn.preprocessing import OrdinalEncoder

# 🎯 Our goal is to predict if the house will be sold for more than 250k USD or not
* We will use categorized price as a **Target** variable

In [2]:
df = pd.read_csv('https://raw.githubusercontent.com/rasvob/VSB-FEI-Fundamentals-of-Machine-Learning-Exercises/master/datasets/zsu_cv1_data.csv', sep=',')
df = df.loc[:, ['SalePrice','MSSubClass','BldgType','HouseStyle','OverallQual','OverallCond','YearBuilt','Heating','CentralAir','GrLivArea','BedroomAbvGr']]
df.loc[:, ['Target']] = (df.SalePrice > 250000).astype(int)
df = df.drop(['SalePrice'], axis=1)

In [ ]:
df.head()

# Take a look at the features
* We will need it to answer the questions

In [ ]:
df.describe()

## Categorial features EDA

In [ ]:
df.describe(exclude=np.number)

### BldgType

In [ ]:
df.BldgType.value_counts()

### HouseStyle

In [ ]:
df.HouseStyle.value_counts()

### Heating

In [ ]:
df.Heating.value_counts()

## Missing values

In [ ]:
df.isna().sum()

## Labels distribution

In [ ]:
df.Target.value_counts()

# ✅ Task (2p)
Complete the following tasks:

1. 📈 Describe what operations you are performing for each of the features
    * Mainly focus on categorical features
      
2. 📌 Answer the following questions:
    * **How many values are missing?** None
    * **How many instances do you have in each of the classes?**
      217 are Target, 1243 are non targets
    * 🔎 **Which metric score do you propose for the classification model performance evaluation?**
      quite heavily skewed => f1 score, just accuracy could be misleading
          
3. ⚡Finish your preprocessing pipeline and split the data into the Input and Output part (i.e. `X` and `y` variables)

4. 🌳 Start with the **Decision Tree**
    * Use 5-fold cross validation
    * 🔎 Will you use *standard* cross validation or *stratified* cross validation?
    * Compute mean of the obtained score values
      
5. 🚀 Select one other algorithm from https://scikit-learn.org/stable/supervised_learning.html
    * Repeat the 5-fold CV
      
6. 📒 **Write down which default model is better**

7. 📊 Experiment with hyper-parameters
    * Select at least one important parameter for the model
    * Set the parameter value range
        * You can use random values, interval of values, ...
    * Do the 5-fold CV
        * Compute mean of the obtained score values
    * Document the experiment results using tables and/or plots
    * Describe the results in a Markdown cell

8. 📒 **Write down which model (default or tuned) is the best and why**

* **Document everything you do in a Markdown cells**
    * ❌ Results interpretation figured in real-time during task check is not allowed! ❌


##One-Hot Encoding
* new binary col for each unique value
* when you are using algoes that work based on ordinality (higher num, higher rank) or distance, and the data cannot be interpreted like that -> for example ports of embarkation

##Label Encoding
* each label gets a number
* like decks labeled A,B,C etc. have nums assigned and transformed into decks 0,1,2



In [3]:
#preprocessing
#one hot encoding all
df = pd.concat([df, pd.get_dummies(df['BldgType'], prefix='BldgType')], axis=1).drop('BldgType', axis=1)
df = pd.concat([df, pd.get_dummies(df['HouseStyle'], prefix='HouseStyle')], axis=1).drop('HouseStyle', axis=1)
df = pd.concat([df, pd.get_dummies(df['Heating'], prefix='Heating')], axis=1).drop('Heating', axis=1)

#except CentralAir, that one is binary Y/N, so ordinal is used
df.CentralAir.value_counts()
ca_cats = ['N', 'Y']
enc_ca = OrdinalEncoder(categories=[ca_cats])
df.loc[:, 'CentralAir'] = enc_ca.fit_transform(df[['CentralAir']])

df

,MSSubClass,OverallQual,OverallCond,YearBuilt,CentralAir,GrLivArea,BedroomAbvGr,Target,BldgType_1Fam,BldgType_2fmCon,...,HouseStyle_2.5Unf,HouseStyle_2Story,HouseStyle_SFoyer,HouseStyle_SLvl,Heating_Floor,Heating_GasA,Heating_GasW,Heating_Grav,Heating_OthW,Heating_Wall
0,60,7,5,2003,1.0,1710,3,0,True,False,...,False,True,False,False,False,True,False,False,False,False
1,20,6,8,1976,1.0,1262,3,0,True,False,...,False,False,False,False,False,True,False,False,False,False
2,60,7,5,2001,1.0,1786,3,0,True,False,...,False,True,False,False,False,True,False,False,False,False
3,70,7,5,1915,1.0,1717,3,0,True,False,...,False,True,False,False,False,True,False,False,False,False
4,60,8,5,2000,1.0,2198,4,0,True,False,...,False,True,False,False,False,True,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1455,60,6,5,1999,1.0,1647,3,0,True,False,...,False,True,False,False,False,True,False,False,False,False
1456,20,6,6,1978,1.0,2073,3,0,True,False,...,False,False,False,False,False,True,False,False,False,False
1457,70,7,9,1941,1.0,2340,4,1,True,False,...,False,True,False,False,False,True,False,False,False,False
1458,20,5,6,1950,1.0,1078,2,0,True,False,...,False,False,False,False,False,True,False,False,False,False


In [4]:
#data spliting into results - X and data - y
X, y = df.loc[:, df.columns != 'Target'], df.loc[:, 'Target'] #X training data, y lbl data


In [5]:
#test and train data split
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2, random_state=13)
X_train.shape, X_test.shape, y_train.shape, y_test.shape

((1168, 26), (292, 26), (1168,), (292,))

In [8]:
import statistics

In [14]:
#util score printer
def print_score(scores):
  print("Scores:")
  for x in scores:
      print(f"{x:.3}")
  print(f"Mean score: {statistics.mean(scores):.3}")
  return

In [16]:
#Tree training
#stratifiedKFold will balance the amount of classes in each fold
#ensuring even distribution of target variable
#so a situtation where one or more folds simply does not have one target class
#(and thus skewing results) does not happen
skf = StratifiedKFold(n_splits=5)
scores = list()
for train_index, test_index in skf.split(X, y):
    X_train, X_test = X.iloc[train_index, :], X.iloc[test_index, :]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]
    clf = DecisionTreeClassifier(random_state=13)
    clf.fit(X_train, y_train)
    y_pred = clf.predict(X_test)
    scores.append(f1_score(y_test, y_pred))
    print(f'Target ratio in train set: {y_train.value_counts(normalize=True)[1]:.2}; Target ratio in test set: {y_test.value_counts(normalize=True)[1]:.2}')

print_score(scores)


Target ratio in train set: 0.15; Target ratio in test set: 0.15
Target ratio in train set: 0.15; Target ratio in test set: 0.15
Target ratio in train set: 0.15; Target ratio in test set: 0.15
Target ratio in train set: 0.15; Target ratio in test set: 0.15
Target ratio in train set: 0.15; Target ratio in test set: 0.15
Scores:
0.667
0.776
0.568
0.733
0.699
Mean score: 0.689


In [17]:
#Forest training also with stratifiedKFold
skf = StratifiedKFold(n_splits=5)
scores = list()
for train_index, test_index in skf.split(X, y):
    X_train, X_test = X.iloc[train_index, :], X.iloc[test_index, :]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]
    clf = RandomForestClassifier(random_state=13)
    clf.fit(X_train, y_train)
    y_pred = clf.predict(X_test)
    scores.append(f1_score(y_test, y_pred))
    print(f'Target ratio in train set: {y_train.value_counts(normalize=True)[1]:.2}; Target ratio in test set: {y_test.value_counts(normalize=True)[1]:.2}')

print_score(scores)


Target ratio in train set: 0.15; Target ratio in test set: 0.15
Target ratio in train set: 0.15; Target ratio in test set: 0.15
Target ratio in train set: 0.15; Target ratio in test set: 0.15
Target ratio in train set: 0.15; Target ratio in test set: 0.15
Target ratio in train set: 0.15; Target ratio in test set: 0.15
Scores:
0.744
0.833
0.703
0.815
0.716
Mean score: 0.762


In [19]:
from scipy.stats import randint, uniform
from sklearn.model_selection import RandomizedSearchCV

##DT hyperparameters
* max_depth - def None - max num of lvls
* min_samples_split - def 2 - min num of samples a node must have before splitting (based on Gini by def), sample being rows, data inputs
* min_samples_leaf - def 1 - min amount of samples req to become a leaf
* criterion - 'gini' - used to measure quality of split
* max_features - def None - limits introduce more randomnes -> for Random Forests and other ensemble methods, features = attributes like Heating or Year built, limiting the amount of them to consider while creating a split increases randomnes

In [41]:
#Hyperparams tuning for Decision Tree
dt_clf = DecisionTreeClassifier(random_state=13)

hyperparams = {
    'min_samples_split' : randint(2, 50), #def (and min) is 2
    'max_depth' : randint(1, 25), #def is None
    'min_samples_leaf' : randint(1, 30) #def is 1
}

search = RandomizedSearchCV(
    estimator = dt_clf,
    param_distributions = hyperparams,
    n_iter = 100, #how many random settings to try
    cv = 5, #hould be def, how many folds, should use Stratified due to binary or
    scoring = 'f1',
    random_state = 13,
    n_jobs = -1 #use all available processors
)

search.fit(X_train, y_train)
print(f"Best f1: {search.best_score_}")
print(search.best_params_)

#test result on clean data
best_dt_clf = search.best_estimator_
y_pred_best = best_dt_clf.predict(X_test)
best_f1 = f1_score(y_test, y_pred_best)

print(f"Best DT f1 score: {best_f1}")

#Drop likely caused by overfitting

Best f1: 0.786866462127373
{'max_depth': 14, 'min_samples_leaf': 11, 'min_samples_split': 11}
Best DT f1 score: 0.6746987951807228


##Random forest
* fits large number of trees and then relies on their vote or avg to decide the final result
* Bagging - training individual trees on different, randomized subsets of data
* Sampling with replacement - some samples might be used multiple times across different subsets, some sample may be never used (Out-of-bag samples)
* Feature selections on split creations are also randomized subsets of all features


In [47]:
df.shape #rows, cols - cols = num of features

(1460, 27)

In [48]:
#Hyperparams tuning for Random Forests
dt_clf = RandomForestClassifier(random_state=13)

hyperparams = {
    'n_estimators' : randint(50, 200), #num of trees, def is 100
    'max_samples' : randint(10, 100),
    'max_features' : randint(1, 27), #def is 1.0 as in all I think
}

search = RandomizedSearchCV(
    estimator = dt_clf,
    param_distributions = hyperparams,
    n_iter = 100, #how many random settings to try
    cv = 5, #hould be def, how many folds, should use Stratified due to binary or
    scoring = 'f1',
    random_state = 13,
    n_jobs = -1 #use all available processors
)

search.fit(X_train, y_train)
print(f"Best f1: {search.best_score_}")
print(search.best_params_)

#test result on clean data
best_dt_clf = search.best_estimator_
y_pred_best = best_dt_clf.predict(X_test)
best_f1 = f1_score(y_test, y_pred_best)

print(f"Best DT f1 score: {best_f1}")

#performance still drops, but not as badly as in single Decision Tree

Best f1: 0.8003900440281514
{'max_features': 23, 'max_samples': 73, 'n_estimators': 163}
Best DT f1 score: 0.7228915662650602


##Results:
* In both cases hyperparameter tuning seems to result in worse performance on new data, but better performance on tests, suggests overfitting, fixing would require more tuning.
* Random forest performs better then single Decision Tree.